# LEVIR-Ship — Journal factorial: detection heads × FTSC

Four fully matched training arms: P345_NOFTSC, P345_FTSC, P23_NOFTSC, P23_FTSC. Three seeds: 42/43/44. This notebook borrows **dataset retrieval, token handling, official YOLOv8n initialization, fork worker settings and canonical Hugging Face archival** from `current_live_training_notebok_LEV.ipynb`.

**Default is safe**: code/data/pretrained setup and static preflight, but *no training and no HF writes*. Source code must be pushed to `TLMHoang-AI/YOLO_FTSC` before a fresh Marimo Live checkout can use it. No token values are stored in this notebook.

When training is enabled, read HF inventory and restore fully verified cases first, then train selected missing cases and **publish each completed run immediately**. Any network failure, partial remote archive, contract mismatch, or incompatible local checkpoint aborts rather than triggering a retrain.

In [ ]:
from pathlib import Path
import os, sys, json
# Match the original LEVIR live notebook transport setting before importing the HF client.
os.environ.setdefault("HF_HUB_DISABLE_XET", "1")

ROOT = Path("/marimo/yolo_code")
GITHUB_REPO_URL = "https://github.com/TLMHoang-AI/YOLO_FTSC.git"
GITHUB_BRANCH = "main"
HF_DATA_REPO = "duyle2408/LevirShipData"
HF_RESULTS_REPO = "TLMHoang/FTSC_results"

# Runner selection: edit these controls only.
SYNC_GITHUB_CODE = True           # safe clone or fast-forward; NO reset --hard or git clean
INSTALL_MISSING_DEPENDENCIES = False
PROMPT_FOR_HF_TOKEN = False       # opt-in: getpass, never store token in notebook source
PREPARE_DATASET = True            # downloads from HF only if the 3,896 pairs are absent
PREPARE_PRETRAINED = True         # downloads official yolov8n.pt if absent
RUN_OFFLINE_TESTS = True
RUN_STATIC_PREFLIGHT = True
RUN_MODEL_PREFLIGHT = False
RUN_HF_PREFLIGHT_AND_RESTORE = False
RUN_TRAINING = False              # EXPLICIT OPT-IN for GPU training + per-seed HF upload
RUN_AGGREGATION = False           # only after all 12 new-cohort runs are complete
ALLOW_RESUME = False             # explicit opt-in after checking last.pt

DEVICE = "cuda:0"
WORKERS = 4
CASES = ["P345_NOFTSC", "P345_FTSC", "P23_NOFTSC", "P23_FTSC"]
SEEDS = [42, 43, 44]
OUTPUT_ROOT = ROOT / "runs/levir_ftsc_head_factorial_v1"
HF_CACHE = Path("/marimo/hf_results_cache/levir_ftsc_head_factorial_v1")
if PROMPT_FOR_HF_TOKEN and not os.environ.get("HF_TOKEN", "").strip():
    from getpass import getpass
    os.environ["HF_TOKEN"] = getpass("Hugging Face access token: ").strip()
HF_TOKEN = os.environ.get("HF_TOKEN", "").strip()  # secret injected at runtime
GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN", "").strip()
print("HF token configured:", bool(HF_TOKEN), "| GitHub token configured:", bool(GITHUB_TOKEN))
print("Training enabled:", RUN_TRAINING, "| chosen arms:", CASES, "| seeds:", SEEDS)

In [ ]:
if INSTALL_MISSING_DEPENDENCIES:
    import subprocess
    packages = ["huggingface_hub", "pyyaml", "pycocotools", "ultralytics-thop",
                "albumentations", "kornia", "timm", "opencv-python-headless", "pillow"]
    subprocess.run([sys.executable, "-m", "pip", "install", *packages], check=True)
else:
    import yaml
    from huggingface_hub import HfApi, snapshot_download
    print("Found Python YAML and Hugging Face clients. Dependency install skipped.")

In [ ]:
# Safe adaptation of source notebook's GitHub checkout: no destructive reset, no token in URL.
if SYNC_GITHUB_CODE:
    import subprocess, base64
    git_env = os.environ.copy()
    git_env["GIT_TERMINAL_PROMPT"] = "0"
    if GITHUB_TOKEN:
        basic = base64.b64encode(("x-access-token:" + GITHUB_TOKEN).encode()).decode()
        git_env["GIT_CONFIG_COUNT"] = "1"
        git_env["GIT_CONFIG_KEY_0"] = "http.https://github.com/.extraheader"
        git_env["GIT_CONFIG_VALUE_0"] = "AUTHORIZATION: basic " + basic
    def git_checked(arguments, cwd=None):
        result = subprocess.run(["git", *arguments], cwd=cwd, env=git_env,
                                text=True, capture_output=True)
        if result.returncode:
            # Never show credential-bearing subprocess environments.
            raise RuntimeError("GitHub checkout failed. Check repository permissions, "
                               "branch, local modifications and GITHUB_TOKEN.")
        return result.stdout.strip()
    if not ROOT.exists():
        ROOT.parent.mkdir(parents=True, exist_ok=True)
        git_checked(["clone", "--branch", GITHUB_BRANCH, "--single-branch",
                     GITHUB_REPO_URL, str(ROOT)])
    elif (ROOT / ".git").is_dir():
        git_checked(["fetch", "origin", GITHUB_BRANCH], cwd=ROOT)
        git_checked(["merge", "--ff-only", f"origin/{GITHUB_BRANCH}"], cwd=ROOT)
    else:
        raise RuntimeError(f"{ROOT} exists but is not a Git repository; refusing to overwrite it.")
    print("YOLO FTSC checkout revision:", git_checked(["rev-parse", "--short", "HEAD"], cwd=ROOT))
else:
    print("GitHub sync disabled; using existing local checkout:", ROOT)
assert ROOT.is_dir(), f"Code checkout missing: {ROOT}"

In [ ]:
import importlib
VENDOR = (ROOT / "models_related/ultralytics").resolve()
required_source = [
    "train_levir_scripts/train_levir_ftsc_head_factorial_v1.py",
    "train_levir_scripts/ftsc_head_factorial_hf_v1.py",
    "models_related/models_config/yolov8/levir/yolov8n_p345_levir_factorial_ftsc.yaml",
    "utils/ftsc_results_uploader.py",
]
missing = [rel for rel in required_source if not (ROOT / rel).is_file()]
if missing:
    raise RuntimeError("Factorial code not found in Marimo Live checkout. Push the "
                       "new local code to GitHub main before starting Live: " + str(missing))
for item in (str(ROOT), str(VENDOR)):
    if item not in sys.path:
        sys.path.insert(0, item)
import ultralytics
if VENDOR not in Path(ultralytics.__file__).resolve().parents:
    raise RuntimeError("Wrong Ultralytics package imported. Restart kernel to load local FTSC fork.")
from train_levir_scripts import train_levir_ftsc_head_factorial_v1 as suite
from train_levir_scripts import ftsc_head_factorial_hf_v1 as hf_suite
from train_levir_scripts import train_all_levir_yolov8n_p2_routing as workflow
print("Verified local FTSC fork and factorial source files")

In [ ]:
# No GPU or HF write: validate paired YAML graphs, training policy, HF archive routing.
if RUN_OFFLINE_TESTS:
    import unittest
    tests = unittest.defaultTestLoader.loadTestsFromNames([
        "train_levir_scripts.test_levir_ftsc_head_factorial_v1",
        "train_levir_scripts.test_ftsc_head_factorial_hf_v1",
    ])
    outcome = unittest.TextTestRunner(verbosity=2).run(tests)
    if not outcome.wasSuccessful():
        raise RuntimeError("Offline regression tests failed; training blocked.")
    print("Offline factorial and HF archival regression tests passed.")

In [ ]:
# Faithful LEVIR dataset retrieval: original notebook checks 3,896 matching pairs.
from huggingface_hub import snapshot_download

def find_data_root():
    candidates = [Path.cwd(), Path.cwd() / "LevirShipData",
                  Path("/marimo"), Path("/marimo/LevirShipData"),
                  ROOT / "LevirShipData"]
    for folder in dict.fromkeys(candidates):
        if not ((folder / "All Images").is_dir()
                and (folder / "All Annotations").is_dir()):
            continue
        images = {p.stem for p in (folder / "All Images").glob("*.png")}
        labels = {p.stem for p in (folder / "All Annotations").glob("*.txt")}
        if len(images) == len(labels) == 3896 and images == labels:
            return folder.resolve()
    return None

DATA_ROOT = find_data_root()
if PREPARE_DATASET and DATA_ROOT is None:
    if not HF_TOKEN:
        raise RuntimeError("Set HF_TOKEN in the Marimo Live environment; missing LEVIR dataset.")
    print("Downloading LEVIR dataset from", HF_DATA_REPO)
    snapshot_download(repo_id=HF_DATA_REPO, repo_type="dataset",
                      token=HF_TOKEN, local_dir=str(Path("/marimo")), max_workers=2)
    DATA_ROOT = find_data_root()
if PREPARE_DATASET:
    assert DATA_ROOT is not None, "LEVIR dataset download incomplete: expected 3,896 image/label pairs"
    from argparse import Namespace
    DATASET_ROOT = ROOT / "datasets"
    data_yaml = DATASET_ROOT / "levir_ship_yolo_seed42/levir_ship.yaml"
    if data_yaml.is_file():
        workflow.validate_split(data_yaml, DATA_ROOT)  # reject changes instead of silently rewriting
    else:
        data_yaml = workflow.prepare_fixed_split(
            Namespace(data_root=DATA_ROOT, dataset_root=DATASET_ROOT, split_seed=42))
    workflow.validate_split(data_yaml, DATA_ROOT)
    print("LEVIR source root:", DATA_ROOT, "| fixed split:", data_yaml)
else:
    DATASET_ROOT = ROOT / "datasets"
    print("Dataset preparation disabled")

In [ ]:
# Official YOLOv8n weight retrieval, using the vendored Ultralytics fork.
PRETRAINED = ROOT / "yolov8n.pt"
if PREPARE_PRETRAINED:
    from ultralytics.utils.downloads import attempt_download_asset
    if not PRETRAINED.is_file():
        fetched = Path(attempt_download_asset(str(PRETRAINED))).resolve()
        if fetched != PRETRAINED.resolve():
            raise RuntimeError(f"Unexpected pretrained checkpoint path: {fetched}")
    assert PRETRAINED.is_file() and PRETRAINED.stat().st_size > 0
    print("Pretrained YOLOv8n ready:", PRETRAINED, PRETRAINED.stat().st_size, "bytes")

In [ ]:
# Original notebook's Linux dataloader fork workaround (required for workers > 0).
if RUN_TRAINING and WORKERS > 0:
    import multiprocessing as mp
    from ultralytics.data import build as data_build
    if "fork" not in mp.get_all_start_methods():
        raise RuntimeError("Multiprocessing fork unavailable. Choose WORKERS = 0.")
    loader = data_build.InfiniteDataLoader
    if not getattr(loader, "_ftsc_fork_context_enabled", False):
        old_init = loader.__init__
        ctx = mp.get_context("fork")
        def init_with_fork(self, *args, **kwargs):
            if kwargs.get("num_workers", 0) > 0:
                kwargs.setdefault("multiprocessing_context", ctx)
            return old_init(self, *args, **kwargs)
        loader.__init__ = init_with_fork
        loader._ftsc_fork_context_enabled = True
    print("Linux fork multiprocessing enabled for FTSC dataset workers")

In [ ]:
COMMON = [
    "--data-root", str(DATA_ROOT if DATA_ROOT is not None else Path("/marimo")),
    "--dataset-root", str(DATASET_ROOT),
    "--pretrained", str(PRETRAINED),
    "--project", str(OUTPUT_ROOT),
    "--device", DEVICE,
    "--workers", str(WORKERS),
]
if RUN_STATIC_PREFLIGHT:
    suite.main(COMMON)
if RUN_MODEL_PREFLIGHT:
    suite.main(COMMON + ["--model-preflight"])

In [ ]:
# HF-first inventory and exact archive recovery; all exceptions deliberately propagate.
# Never train based on failed network reads or a partial/colliding remote archive.
hf_uploader = None
hf_report = None
if RUN_HF_PREFLIGHT_AND_RESTORE or RUN_TRAINING:
    if not HF_TOKEN:
        raise RuntimeError("HF_TOKEN is required before training for immediate checkpoint archival.")
    if DATA_ROOT is None or not PRETRAINED.is_file():
        raise RuntimeError("Dataset and official pretrained checkpoint must be ready first.")
    assert data_yaml.is_file()
    args = suite.parse_args(COMMON)
    args.pretrained = PRETRAINED.resolve()
    args.data_root = DATA_ROOT.resolve()
    args.dataset_root = DATASET_ROOT.resolve()
    args.project = OUTPUT_ROOT.resolve()
    source = suite.source_preflight()
    hf_report = hf_suite.inventory(HF_TOKEN)
    print("HF inventory revision:", hf_report["revision"])
    print("Existing completed HF cells:",
          [(x["case"], x["seed"]) for x in hf_report["rows"] if x["state"] == "MARKED_PRESENT"])
    hf_uploader = hf_suite.make_uploader(HF_TOKEN, HF_CACHE)
    recovered = hf_suite.restore_available(
        hf_uploader, hf_report, args, data_yaml, source)
    print("Verified restored HF cells:", recovered)

In [ ]:
# Train one case/seed at a time so that each completed checkpoint is pushed immediately.
# Existing same-contract local or HF runs are reused only after SHA/manifest validation.
if RUN_TRAINING:
    if hf_uploader is None:
        raise RuntimeError("HF preflight not complete; refusing unarchived training.")
    for seed in SEEDS:
        for case in CASES:
            print(f"[FTSC 2x2] Processing {case}/seed_{seed}", flush=True)
            cmd = COMMON + ["--confirm-run", "--cases", case, "--seeds", str(seed)]
            if ALLOW_RESUME:
                cmd.append("--resume")
            suite.main(cmd)  # may train, evaluate or verify a pre-existing complete run
            result = hf_suite.publish_or_verify(
                hf_uploader, HF_TOKEN, args, case, seed, data_yaml, source)
            print("[FTSC 2x2] HF archival:", result, flush=True)
    print("All requested factorial cells processed and archived")

In [ ]:
if RUN_AGGREGATION:
    if not data_yaml.is_file():
        raise RuntimeError("Frozen LEVIR split missing")
    # Requires full 2 x 2 x 3 completed run cohort; rejects partial summaries.
    suite.main(COMMON + ["--aggregate"])
    for output in ("factorial_runs.csv", "factorial_aggregate.csv", "factorial_effects.csv"):
        assert (OUTPUT_ROOT / output).is_file(), f"Missing result: {output}"
    print("Factorial result tables:", OUTPUT_ROOT)

## Run order and recovery

**First launch**: leave RUN_TRAINING=False, run code sync, dataset preparation and static preflight. Set RUN_MODEL_PREFLIGHT=True for a subsequent run and verify four Detect stride sets and FTSC assignment.

**Pilot**: set RUN_TRAINING=True with `CASES=["P345_FTSC"]` and `SEEDS=[42]`. This performs real training and writes to HF as soon as that seed finishes. On success expand the configuration list to all four arms and three seeds. Completed, matching runs will be verified/reused; different or partial runs fail closed.

**Aggregation**: after 12/12 runs, enable RUN_AGGREGATION=True (can leave RUN_TRAINING=False). This produces three CSV outputs containing both FTSC-within-head effects and difference-in-differences.

**Security**: `HF_TOKEN` and optional `GITHUB_TOKEN` should be supplied as environment secrets in Marimo Live. Alternatively set PROMPT_FOR_HF_TOKEN=True to enter the HF token once via getpass at runtime; this never embeds the value in source. Never paste literal credentials into notebook cells. The *source* training notebook contains a literal HF token and should have that credential revoked/rotated.